# Gmail Bot with AI-Generated Emails

Send personalized AI-generated emails with attachments using data from Excel.

## Setup Requirements
1. Gmail: Enable 2FA → Generate App Password
2. AI: Get API key from OpenAI or Google AI Studio

---
## 1. Install Packages

In [ ]:
!pip install pandas openpyxl openai google-generativeai

---
## 2. Imports

In [ ]:
# Standard library
import smtplib
import os
import time

# Email handling
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Data processing
import pandas as pd

# Colab file upload
from google.colab import files

# AI - Choose one:
import openai                      # Option A: OpenAI (ChatGPT)
import google.generativeai as genai  # Option B: Google Gemini

---
## 3. Configuration

In [ ]:
# Gmail credentials
SENDER_EMAIL = "your_email@gmail.com"
APP_PASSWORD = "xxxx xxxx xxxx xxxx"  # From Google Account → Security → App Passwords

# AI API Keys (use one)
OPENAI_API_KEY = "sk-..."           # From platform.openai.com/api-keys
GEMINI_API_KEY = "AIza..."          # From aistudio.google.com/apikey

# Choose AI provider: "openai" or "gemini"
AI_PROVIDER = "gemini"

# Your details for email personalization
SENDER_NAME = "Your Name"
SENDER_COMPANY = "Your Company"
EMAIL_PURPOSE = "partnership for our news aggregation platform"

---
## 4. Upload Files

In [ ]:
# Upload Excel file
print("Upload Excel file (columns: 'email', 'company_name')")
uploaded = files.upload()
excel_file = list(uploaded.keys())[0]

# Upload attachments
print("\nUpload attachment(s)")
attachments = files.upload()
attachment_files = list(attachments.keys())

---
## 5. Read Excel Data

In [ ]:
def read_excel_data(file_path):
    """Read email and company data from Excel."""
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.lower().str.strip()
    return df[['email', 'company_name']].dropna()

df = read_excel_data(excel_file)
print(f"Found {len(df)} recipients:")
df.head()

---
## 6. AI Email Generator

In [ ]:
# Initialize AI client
if AI_PROVIDER == "openai":
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
else:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')

In [ ]:
def generate_email_content(company_name, purpose, sender_name, sender_company):
    """Use AI to generate personalized email content."""
    
    prompt = f"""
    Write a professional business email with these details:
    - Recipient company: {company_name}
    - Purpose: {purpose}
    - Sender: {sender_name} from {sender_company}
    
    Requirements:
    - Keep it concise (3-4 paragraphs)
    - Professional but friendly tone
    - Include a clear call to action
    - Mention there's an attachment with more details
    - Do NOT include subject line, just the body
    """
    
    if AI_PROVIDER == "openai":
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500
        )
        return response.choices[0].message.content
    else:
        response = model.generate_content(prompt)
        return response.text


def generate_subject(company_name, purpose):
    """Use AI to generate email subject line."""
    
    prompt = f"Write a short, compelling email subject line for a {purpose} email to {company_name}. Return only the subject line, nothing else."
    
    if AI_PROVIDER == "openai":
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        return response.choices[0].message.content.strip()
    else:
        response = model.generate_content(prompt)
        return response.text.strip()

In [ ]:
# Test AI generation
test_company = df.iloc[0]['company_name']
print(f"Testing AI for: {test_company}\n")

subject = generate_subject(test_company, EMAIL_PURPOSE)
print(f"Subject: {subject}\n")

body = generate_email_content(test_company, EMAIL_PURPOSE, SENDER_NAME, SENDER_COMPANY)
print(f"Body:\n{body}")

---
## 7. Email Functions

In [ ]:
def create_email_with_attachment(sender, recipient, subject, body, attachment_paths):
    """Create MIME email with attachments."""
    msg = MIMEMultipart()
    msg['From'] = sender
    msg['To'] = recipient
    msg['Subject'] = subject
    
    msg.attach(MIMEText(body, 'plain'))
    
    for file_path in attachment_paths:
        with open(file_path, 'rb') as f:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(f.read())
            encoders.encode_base64(part)
            part.add_header(
                'Content-Disposition',
                f'attachment; filename="{os.path.basename(file_path)}"'
            )
            msg.attach(part)
    
    return msg

In [ ]:
def send_emails_with_ai(df, sender_email, app_password, attachment_files, delay=3):
    """Send AI-generated emails to all recipients."""
    
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(sender_email, app_password.replace(' ', ''))
    
    sent, failed = 0, []
    
    for idx, row in df.iterrows():
        try:
            email = row['email']
            company = row['company_name']
            
            # Generate AI content
            print(f"Generating email for {company}...")
            subject = generate_subject(company, EMAIL_PURPOSE)
            body = generate_email_content(company, EMAIL_PURPOSE, SENDER_NAME, SENDER_COMPANY)
            
            # Create and send
            msg = create_email_with_attachment(sender_email, email, subject, body, attachment_files)
            server.sendmail(sender_email, email, msg.as_string())
            
            sent += 1
            print(f"✓ Sent to {company} ({email})")
            time.sleep(delay)
            
        except Exception as e:
            failed.append({'email': email, 'company': company, 'error': str(e)})
            print(f"✗ Failed: {email} - {e}")
    
    server.quit()
    print(f"\n--- Done: {sent}/{len(df)} sent ---")
    return sent, failed

---
## 8. Send Emails

In [ ]:
# Test with first recipient (uncomment to run)
# test_df = df.head(1)
# sent, failed = send_emails_with_ai(test_df, SENDER_EMAIL, APP_PASSWORD, attachment_files)

# Send to all (uncomment to run)
# sent, failed = send_emails_with_ai(df, SENDER_EMAIL, APP_PASSWORD, attachment_files)

---
## Quick Reference: All Imports

```python
# Core
import smtplib
import os
import time

# Email
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Data
import pandas as pd

# Colab
from google.colab import files

# AI (pick one)
import openai
import google.generativeai as genai
```